<a href="https://colab.research.google.com/github/antoine-bourgois/MIA_2026_UE16_NLP/blob/main/code/2_Cooccurrence_Tokens_Vecteurs.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## TP 2 : Représentation Tokens - Matrice de cooccurrence

- Charger un Dataset
- Description statistique du dataset
- Matrice de cooccurrence
- Mots les plus similaires
- Réduction de dimension
- Visualisation
- Évaluation modèle de similarité sur benchmarks

### Installation des librairies Python utiles pour le TP

In [ ]:
!pip install tqdm jupyter ipywidgets datasets pyarrow huggingface_hub pandas matplotlib numpy scipy scikit-learn umap-learn plotly --quiet

In [ ]:
from tqdm.auto import tqdm
from collections import Counter

### Charger un jeu de données

In [ ]:
from datasets import load_dataset_builder

dataset_path = "AntoineBourgois/Gallica_Hors-d-Usage_cleaned"

builder = load_dataset_builder(dataset_path)

builder = load_dataset_builder(dataset_path, )

print(f"Le jeu de données {dataset_path!r} contient {len(builder.info.splits)} sous-ensembles :")
for name, split in builder.info.splits.items():
    print(f"    {name!r:<20}{split.num_examples:<6,} documents")

In [ ]:
# sous_ensemble, label_column = "auteur_genre", "auteur_genre"
# sous_ensemble, label_column = "auteur", "auteur"
sous_ensemble, label_column = "date_publication", "date_publication"
# sous_ensemble, label_column = "genre_litteraire", "genre_litteraire"

import pyarrow.parquet as pq
from huggingface_hub import hf_hub_download

dataset_path = "AntoineBourgois/Gallica_Hors-d-Usage_cleaned"

fichier_local = hf_hub_download(
    repo_id=dataset_path,
    filename=f"data/{sous_ensemble}-00000-of-00001.parquet",
    repo_type="dataset",
    force_download=True,
)

table = pq.read_table(fichier_local)
dataset = table.to_pylist()

print("Nombre de documents :", len(dataset))


def conversions_string_vers_liste(string):
    if isinstance(string, str):
        return string.split(" ")


for document in dataset:
    document["tokens"] = conversions_string_vers_liste(document["tokens"])
    document["lemmes"] = conversions_string_vers_liste(document["lemmes"])

### Structure du dataset

In [ ]:
print(f"Type dataset: {type(dataset)}")
print(f"Longueur dataset: {len(dataset)}")

doc_0 = dataset[0]
print(f"Type 1er document: {type(doc_0)}")
print(f"Clés d'un document: {doc_0.keys()}")

In [ ]:
for key, value in doc_0.items():
    print(f"{key:<20} {type(value)}    {str(value)[:100]}")

### Vocabulaire

Construire un dictionnaire du vocabulaire du corpus.

### Choix de l'unité élémentaire : tokens ou lemmes ?

In [ ]:
for word_type in ["tokens", "lemmes"]:
    all_words = [word.lower()
                  for document in dataset
                  for word in document[word_type]
                  ]
    vocab = sorted(set(all_words))
    print(f"{word_type:<10}  Taille du vocabulaire: {len(vocab):,} words")

Nous retenons les lemmes pour réduire la taille du vocabulaire

In [ ]:
word_type = "lemmes" # ["lemmes", "tokens"]

In [ ]:
all_words = [word.lower()
                  for document in dataset
                  for word in document[word_type]
                  ]
vocab = sorted(set(all_words))


print(f"Documents : {len(dataset):,}")
print(f"Taille du corpus : {len(all_words):,}")
print(f"Mots / Document : {int(len(all_words)/len(dataset)):,}")
print(f"Taille du vocabulaire : {len(vocab):,} ({word_type})")

In [ ]:
import pandas as pd
from collections import Counter

vocab_counter = Counter(all_words).most_common()
# vocab = list(dict(vocab_counter).keys())

df = pd.DataFrame([
    {
        "Rang": i + 1,
        "Word": word,
        "Occurrences": count,
        "Fréquence (%)": 100 * count / len(all_words)
    }
    for i, (word, count) in enumerate(vocab_counter)
])

df

In [ ]:
word_lists = [
    [word.lower() for word in doc[word_type]]
    for doc in dataset
]
word_counts = Counter(all_words)

print(f"Vocabulaire : {len(vocab_counter):,}")
print(f"Mots apparaissant une seule fois : {sum(c[1] == 1 for c in vocab_counter):,}")
print(f"Mots apparaissant 5 fois ou plus : {sum(c[1] >= 5 for c in vocab_counter):,}")
print(f"Mots apparaissant 20 fois ou plus : {sum(c[1] >= 20 for c in vocab_counter):,}")
print(f"Mots apparaissant 100 fois ou plus : {sum(c[1] >= 100 for c in vocab_counter):,}")
print(f"Mots apparaissant 1 000 fois ou plus : {sum(c[1] >= 1000 for c in vocab_counter):,}")

In [ ]:
import numpy as np
from scipy.sparse import coo_matrix
from tqdm import tqdm

# Context Window
N = 3

word_to_id = {word: i for i, word in enumerate(vocab)}

rows = []
cols = []

for tokens in tqdm(word_lists):

    ids = np.fromiter(
        (word_to_id[w] for w in tokens if w in word_to_id),
        dtype=np.int32
    )

    # Generate context pairs for each distance
    for d in range(1, N + 1):

        if len(ids) <= d:
            continue

        left = ids[:-d]
        right = ids[d:]

        # Both directions: target -> context and context -> target
        rows.append(np.concatenate((left, right)))
        cols.append(np.concatenate((right, left)))

# Concatenate all pairs
rows = np.concatenate(rows)
cols = np.concatenate(cols)

# Build sparse matrix
cooc_matrix = coo_matrix(
    (
        np.ones(len(rows), dtype=np.int32),
        (rows, cols)
    ),
    shape=(len(vocab), len(vocab))
).tocsr()

print("Dimensions de la matrice :", cooc_matrix.shape)
print(f"Nombre d'entrées non nulles : {cooc_matrix.nnz:,} ({cooc_matrix.nnz/(cooc_matrix.shape[0]*cooc_matrix.shape[1]):.4%})")

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

def get_most_similar(
    word,
    matrix,
    word_to_id,
    vocab,
    word_counts,
    top_n=10,
    min_count=1
):

    if word not in word_to_id:
        print(f"'{word}' is not in the vocabulary.")
        return

    word_id = word_to_id[word]

    # Compute similarity against all word vectors
    similarities = cosine_similarity(
        matrix[word_id],
        matrix
    ).flatten().tolist()

    # Rank candidates
    closest_ids = np.argsort(similarities)[::-1]

    results = []

    for idx in closest_ids:

        candidate = vocab[idx]

        # Exclude the query word
        if idx == word_id:
            continue

        # Apply frequency threshold
        if word_counts[candidate] < min_count:
            continue

        results.append((candidate, round(similarities[idx], 4)))

        if len(results) == top_n:
            break

    return results

In [ ]:
min_word_occurrences = 0

filtered_vocab = [
    word for word, count in word_counts.items()
    if count >= min_word_occurrences
]
print(f"Vocabulaire après filtrage: {len(filtered_vocab):,}")

In [ ]:
target_word = "france"
most_similar = get_most_similar(
    target_word,
    cooc_matrix,
    word_to_id,
    vocab,
    word_counts,
    top_n=10,
    min_count=min_word_occurrences
)
print(f"Mot Cible  : {target_word}")
print(f"Contexte N : {N}")
print(f"------------------")
print(pd.DataFrame(most_similar, columns=["Word", "Cos_Sim"]).to_string(index=False, header=False, justify=["right", "center"]))

In [ ]:
for target_word in ["lundi", "fusil", "joue", "maison", "roi", "tristesse", "sept", "france", "robe", "car"]:
   most_similar = get_most_similar(
    target_word,
    cooc_matrix,
    word_to_id,
    vocab,
    word_counts,
    top_n=5,
    min_count=0)
   print(f"{target_word:<10} -> {most_similar[0][0]:<10} ->  {most_similar[1][0]:<10} ->  {most_similar[2][0]:<10} ->  {most_similar[3][0]:<10} ->  {most_similar[4][0]:<10}")

## Réduction de dimension et Visualisation 2D

In [ ]:
min_word_occurrences = 500 # filtre le vocabulaire à garder pour la visualisation

filtered_vocab = [
    word for word, count in word_counts.items()
    if count >= min_word_occurrences
]

print(f"Vocabulaire après filtrage: {len(filtered_vocab):,}")

In [ ]:
# Original vocabulary indices of retained words
filtered_ids = [
    word_to_id[word]
    for word in filtered_vocab
]
filtered_word_to_id = {word: i for i, word in enumerate(filtered_vocab)}

# Filter rows AND columns
filtered_matrix = cooc_matrix[filtered_ids, :][:]
filtered_matrix = filtered_matrix.astype("float32")

print("Original matrix:", cooc_matrix.shape)
print("Filtered matrix:", filtered_matrix.shape)

### Réduction de dimensions avec UMAP

In [ ]:
import umap

reducer = umap.UMAP(
    n_components=2,
    n_neighbors=25,
    min_dist=0.1,
    metric="cosine",
    n_jobs=-1,
    # random_state=42
)

embedding = reducer.fit_transform(filtered_matrix)

# 2. Mean-center the entire 3D projection around (0, 0, 0)
embedding = embedding - embedding.mean(axis=0)

In [ ]:
import pandas as pd
import plotly.express as px

embedding_df = pd.DataFrame({
    "x": embedding[:, 0],
    "y": embedding[:, 1],
    f"{word_type}": filtered_vocab
})

In [ ]:
target_words = ["père", "mère", "rêver"]

embedding_df["group"] = embedding_df[word_type].isin(target_words).map({
    True: "Target",
    False: "Other words"
})

In [ ]:
fig = px.scatter(
    embedding_df,
    x="x",
    y="y",
    color="group",
    hover_name=f"{word_type}",
    title="UMAP Visualization of French Word Co-occurrences",
    opacity=0.8,
)

fig.update_traces(
    marker=dict(size=5)
)

fig.update_layout(
    width=1000,
    height=800,
    template="plotly_white",
    xaxis_title="UMAP 1",
    yaxis_title="UMAP 2",
)

fig.show()

### Similarité en 3 dimensions

In [ ]:
min_word_occurrences = 500

filtered_vocab = [
    word for word, count in word_counts.items()
    if count >= min_word_occurrences
]

print(f"Vocabulary after filtering: {len(filtered_vocab):,}")
# Original vocabulary indices of retained words
filtered_ids = [
    word_to_id[word]
    for word in filtered_vocab
]

# Filter rows AND columns
filtered_matrix = cooc_matrix[filtered_ids, :][:, filtered_ids]
filtered_matrix = filtered_matrix.astype("float32")

print("Original matrix:", cooc_matrix.shape)
print("Filtered matrix:", filtered_matrix.shape)


import umap

reducer = umap.UMAP(
    n_components=3,
    n_neighbors=25,
    min_dist=0.8,
    metric="cosine",
    n_jobs=-1,
    # random_state=42
)

embedding = reducer.fit_transform(filtered_matrix)
embedding = embedding - embedding.mean(axis=0)


import pandas as pd
import plotly.express as px

embedding_df = pd.DataFrame({
    "x": embedding[:, 0],
    "y": embedding[:, 1],
    "z": embedding[:, 2],
    f"{word_type}": filtered_vocab
})

In [ ]:
import numpy as np
import pandas as pd
import umap
from sklearn.preprocessing import normalize

# 1. Filter vocabulary and co-occurrence matrix
min_word_occurrences = 500
filtered_vocab = [w for w, c in word_counts.items() if c >= min_word_occurrences]
filtered_ids = [word_to_id[w] for w in filtered_vocab]
filtered_matrix = cooc_matrix[filtered_ids, :][:, filtered_ids].astype("float32")

print(f"Vocabulary after filtering: {len(filtered_vocab):,}")
print(f"Matrix shape: {cooc_matrix.shape} -> {filtered_matrix.shape}\n")

# 2. UMAP dimensionality reduction + Mean-centering
reducer = umap.UMAP(
    n_components=3,
    n_neighbors=25,
    min_dist=0.8,
    metric="cosine",
    n_jobs=-1,
)
embedding = reducer.fit_transform(filtered_matrix)
embedding -= embedding.mean(axis=0)

embedding_df = pd.DataFrame({
    "x": embedding[:, 0],
    "y": embedding[:, 1],
    "z": embedding[:, 2],
    word_type: filtered_vocab
})

# 3. Extract target words, normalize, and compute cosine similarity
target_words = ["père", "mère", "rêver"]
target_df = embedding_df[embedding_df[word_type].isin(target_words)]

words = target_df[word_type].values
vectors = target_df[["x", "y", "z"]].values
norm_vectors = normalize(vectors, norm="l2")

print("--- Centered & Normalized Vectors (3D) ---")
for word, vec in zip(words, norm_vectors):
    print(f"{word:<8}: {np.round(vec, 4)}")

sim_df = pd.DataFrame(norm_vectors @ norm_vectors.T, index=words, columns=words)

print("\n--- Cosine Similarity Matrix ---")
print(sim_df.round(4))

In [ ]:
sim_df

## Évaluer un modèle de similarité

In [ ]:
import pandas as pd

similarity_benchmarks = {
    "MC": "https://raw.githubusercontent.com/siabar/Multilingual_Wordpairs/master/Gold_Standards/fr-mc.dataset",
    "RG": "https://raw.githubusercontent.com/siabar/Multilingual_Wordpairs/master/Gold_Standards/fr-rg.dataset",
    "SimLex": "https://raw.githubusercontent.com/siabar/Multilingual_Wordpairs/master/Gold_Standards/fr-simlex.dataset",
    "WS353": "https://raw.githubusercontent.com/siabar/Multilingual_Wordpairs/master/Gold_Standards/fr-ws353.dataset",
}

datasets = {}

for name, url in similarity_benchmarks.items():
    df = pd.read_csv(url, sep=";")

    df["dataset"] = name
    df = df[["dataset", "word1", "word2", "score"]].copy()

    df["word1"] = df["word1"].str.lower()
    df["word2"] = df["word2"].str.lower()

    # Ensure word1 (A) comes before word2 (B) alphabetically
    df[["word1", "word2"]] = df[["word1", "word2"]].apply(
        lambda row: sorted(row), axis=1, result_type="expand"
    )

    datasets[name] = df

    print(f"{name}: {df.shape}")

# Merge all datasets into a single DataFrame
df_all = pd.concat(datasets.values(), ignore_index=True)

# Remove duplicate word pairs (regardless of dataset)
# df_all = df_all.drop_duplicates(subset=["word1", "word2"], keep="first")

# Reset index
df_all = df_all.reset_index(drop=True)

print(f"All datasets merged: {df_all.shape}")

In [ ]:
df_all

In [ ]:
from scipy.stats import spearmanr
from sklearn.metrics.pairwise import cosine_similarity

def predict_similarity(word1, word2, matrix, word_to_id):

    if word1 not in word_to_id or word2 not in word_to_id:
        return np.nan

    id1 = word_to_id[word1]
    id2 = word_to_id[word2]

    similarity = cosine_similarity(
        matrix[id1],
        matrix[id2]
    )[0, 0]

    return float(similarity)

def evaluate_similarity(dataset, matrix, word_to_id):

    human_scores = []
    model_scores = []

    for _, row in dataset.iterrows():

        word1 = row["word1"]
        word2 = row["word2"]

        if word1 not in word_to_id or word2 not in word_to_id:
            continue

        id1 = word_to_id[word1]
        id2 = word_to_id[word2]

        similarity = cosine_similarity(
            matrix[id1],
            matrix[id2]
        )[0, 0]

        human_scores.append(row["score"])
        model_scores.append(float(similarity))

    correlation, p_value = spearmanr(
        human_scores,
        model_scores
    )

    coverage = len(human_scores) / len(dataset)

    return correlation, p_value, coverage

In [ ]:
results = {}

for name, dataset in datasets.items():

    correlation, p_value, coverage = evaluate_similarity(
        dataset,
        cooc_matrix,
        word_to_id
    )

    results[name] = {
        "Spearman": correlation,
        "p-value": p_value,
        "Coverage": coverage
    }

results_df = pd.DataFrame(results).T

print(results_df.round(4))

In [ ]:
df_all["prediction"] = df_all.apply(
    lambda row: predict_similarity(
        row["word1"],
        row["word2"],
        cooc_matrix,
        word_to_id
    ),
    axis=1
)

In [ ]:
from sklearn.preprocessing import MinMaxScaler

df_all["score_scaled"] = (
    df_all.groupby("dataset")["score"]
    .transform(lambda x: MinMaxScaler().fit_transform(x.values.reshape(-1, 1)).ravel())
)

df_all["prediction_scaled"] = (
    df_all.groupby("dataset")["prediction"]
    .transform(lambda x: MinMaxScaler().fit_transform(x.values.reshape(-1, 1)).ravel())
)

In [ ]:
df_all["error"] = (
    df_all["score_scaled"] - df_all["prediction_scaled"]
)

df_all["absolute_error"] = df_all["error"].abs()

df_all["squared_error"] = df_all["error"] ** 2

In [ ]:
error_results = df_all.groupby("dataset").agg(
    MAE=("absolute_error", "mean"),
    MSE=("squared_error", "mean"),
    RMSE=("squared_error", lambda x: x.mean() ** 0.5),
    N=("absolute_error", "count")
)

print(error_results.round(4))

In [ ]:
df_all.sort_values(by="absolute_error", ascending=False)

In [ ]:
datasets["SimLex"]

### Simlex encode strictement la synonymie, et non simpement l'appartenance au même champ lexical.

### Limite des approches distributionnelles (cooccurrence)

In [ ]:
datasets["WS353"]